In [ ]:
!pip install tensorflow matplotlib pillow

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

print("TensorFlow Sürümü:", tf.__version__)

In [ ]:
# ImageNet verisetiyle eğitilmiş MobileNetV2 modelini yüklüyoruz
model = MobileNetV2(weights='imagenet')

# Test için TensorFlow'un örnek veri deposundan bir resim indirelim
img_url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg'
img_path = tf.keras.utils.get_file('labrador.jpg', img_url)

# Resmi modelin beklediği boyuta (224x224) getirip yüklüyoruz
img = image.load_img(img_path, target_size=(224, 224))

# Resmi görselleştirelim
plt.imshow(img)
plt.axis('off')
plt.title("Analiz Edilecek Görüntü")
plt.show()

# Resmi modelin anlayabileceği formata (dizi ve batch formatı) çeviriyoruz
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

In [ ]:
# Model üzerinden tahminleri alıyoruz
predictions = model.predict(x)

# Tahminleri okunabilir etiketlere (isim, olasılık) çeviriyoruz (En olası 10 tahmini alalım)
decoded_predictions = decode_predictions(predictions, top=10)[0]

def esik_analizi_yap(tahminler, esik_degeri):
    """
    Verilen tahmin listesini belirli bir güven eşiğine göre filtreler.
    """
    print(f"\n--- GÜVEN EŞİĞİ: %{esik_degeri * 100} ({esik_degeri}) ---")
    
    # Sadece olasılığı eşik değerinden büyük veya eşit olanları seç
    gecen_tahminler = [tahmin for tahmin in tahminler if tahmin[2] >= esik_degeri]
    
    if not gecen_tahminler:
        print("Uyarı: Hiçbir tahmin bu eşik değerini geçemedi!")
    else:
        for i, (imagenet_id, etiket, olasilik) in enumerate(gecen_tahminler):
            print(f"{i+1}. Etiket: {etiket.upper()} - Güven: %{olasilik*100:.2f}")

In [ ]:
# 1. Senaryo: Çok Düşük Eşik (Model ne bulursa getirsin - Yüksek Duyarlılık/Recall)
esik_analizi_yap(decoded_predictions, 0.01)

# 2. Senaryo: Orta Seviye Eşik (Dengeli bir yaklaşım)
esik_analizi_yap(decoded_predictions, 0.15)

# 3. Senaryo: Yüksek Eşik (Model sadece çok emin olduklarını göstersin - Yüksek Kesinlik/Precision)
esik_analizi_yap(decoded_predictions, 0.80)

# 4. Senaryo: Aşırı Yüksek Eşik (Neredeyse imkansız)
esik_analizi_yap(decoded_predictions, 0.99)